# Elasticsearch Basics in Python

This notebook demonstrates how to connect to Elasticsearch from Python, check the connection, create an index, insert documents, search, and delete data.

**Make sure your Elasticsearch and Jupyter containers are running!**

In [2]:
!pip install "elasticsearch>=8.0.0,<10.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.4/979.4 kB 2.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [elasticsearch]0m [elasticsearch]


In [4]:
from elasticsearch import Elasticsearch

# Connect to Elasticsearch running in Docker
es = Elasticsearch(
    'http://elasticsearch:9200',
    basic_auth=('elastic', '111')  # Replace with your actual password
)

# Test connection
if es.ping():
    print('Connected to Elasticsearch!')
else:
    print('Could not connect to Elasticsearch.')

Connected to Elasticsearch!


## Create an Index
Let's create a simple index called `test-index`.

In [5]:
index_name = 'test-index'

# Create the index (ignore if it exists)
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name)
    print(f'Index `{index_name}` created.')
else:
    print(f'Index `{index_name}` already exists.')

Index `test-index` created.


## Index (Insert) a Document
Let's add a simple document to our index.

In [6]:
es.index(index="movies", id=1, document={"title": "Inception",    "director": "Christopher Nolan", "description": "A thief using dream-sharing technology."})
es.index(index="movies", id=2, document={"title": "Interstellar", "director": "Christopher Nolan", "description": "Explorers travel through space using advanced technology."})
es.indices.refresh(index="movies")
print("✅ Documents prêts")

✅ Documents prêts


## 🔎 Recherche de Documents

Elasticsearch utilise un **index inversé** : chaque mot est associé à la liste des documents où il apparaît.

- **Mot partagé** → le mot existe dans plusieurs documents → **plusieurs résultats**
- **Mot unique** → le mot n'existe que dans un document → **un seul résultat**

Chaque résultat a un **`_score`** : c'est le score de pertinence. Plus il est élevé, plus le document correspond à la recherche.

In [7]:
# ─── MOT PARTAGÉ ────────────────────────────────────────────────────────────
# "Nolan" apparaît dans le champ "director" des 2 documents (Inception ET Interstellar)
# L'index inversé ressemble à : { "nolan": [doc_id_1, doc_id_2] }
# → Elasticsearch retourne les 2 documents, triés par _score (pertinence)

res = es.search(index="movies", query={"match": {"director": "Nolan"}})

print(f"🔎 Recherche 'Nolan' → {res['hits']['total']['value']} résultat(s)")
for h in res["hits"]["hits"]:
    # _score : score de pertinence calculé automatiquement (TF-IDF)
    # Plus le score est élevé, plus le document est pertinent
    print(f"   [{h['_score']:.2f}] {h['_source']['title']}")

🔎 Recherche 'Nolan' → 2 résultat(s)
   [0.18] Inception
   [0.18] Interstellar


In [8]:
# ─── MOT UNIQUE ─────────────────────────────────────────────────────────────
# "dream" apparaît UNIQUEMENT dans la description d'Inception
# L'index inversé ressemble à : { "dream": [doc_id_1] }
# → Elasticsearch retourne 1 seul document, sans avoir scanné les autres

res = es.search(index="movies", query={"match": {"description": "dream"}})

print(f"🔎 Recherche 'dream' → {res['hits']['total']['value']} résultat(s)")
for h in res["hits"]["hits"]:
    # Un seul résultat attendu : Inception
    print(f"   [{h['_score']:.2f}] {h['_source']['title']}")

🔎 Recherche 'dream' → 1 résultat(s)
   [0.72] Inception


## Delete the Index
Let's clean up by deleting the index.

In [9]:
es.indices.delete(index="movies")
print(f'Index `{"movies"}` deleted.')

Index `movies` deleted.
